In [559]:
#how does proximity to financial services, job density, population density, education level, and employment affect household income.

In [560]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
 
import scipy.stats as st
import statsmodels.api as sm 
import pylab as py 

# here are some of the tools we will use for our analyses
from sklearn.linear_model import LinearRegression
from sklearn.metrics import PredictionErrorDisplay
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score


import itertools
from itertools import combinations

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression


from functools import partial
from sklearn.model_selection import \
     (cross_validate,
      KFold,
      ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm


import csv
import sqlite3
from pygam import LogisticGAM

from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score



In [561]:
income = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Household Income.csv", na_values=['NA'])
prox_bank = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Prox_bank_csv.csv", na_values=['NA'])
employment = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Employment.csv", na_values=['NA'])
#job_density = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Job_Density_csv.csv", na_values=['NA'])
#pop_density = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Population_Density_csv.csv", na_values=['NA'])
education = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Education Level - Bachelor's Degree.csv", na_values=['NA'])
age = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Age_csv.csv", na_values=['NA'])
home_owner = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\home_owner_csv.csv", na_values=['NA'])

In [562]:
conn = sqlite3.connect("my_data.db")

income.to_sql("income", conn, if_exists="replace", index=False)
prox_bank.to_sql("prox_bank", conn, if_exists="replace", index=False)
employment.to_sql("employment", conn, if_exists="replace", index=False)
education.to_sql("education", conn, if_exists="replace", index=False) 
age.to_sql("age", conn, if_exists="replace", index=False)
home_owner.to_sql("home_owner", conn, if_exists="replace", index=False)

459

In [563]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE income(
    NPA INT PRIMARY KEY,
    median_income DOUBLE
)
""")

cursor.execute("""
CREATE TABLE prox_bank(
    NPA INTEGER,
    "2023" DOUBLE
)
""")

cursor.execute("""
CREATE TABLE employment(
    NPA INTEGER,
    employment_rate DOUBLE
)
""")

cursor.execute("""
CREATE TABLE education(
    NPA INTEGER,
    bachelors_percent DOUBLE
)
""")
cursor.execute("""
CREATE TABLE age(
    NPA INTEGER,
    median_age DOUBLE
)
""")
cursor.execute("""
CREATE TABLE home_owner(
    NPA INTEGER,
    home_owner_percent DOUBLE
    )
    """)

In [564]:
for _, row in income.iterrows():
    cursor.execute("INSERT INTO income VALUES (?, ?)",
                   (row['NPA'], row['2023']))

for _, row in prox_bank.iterrows():
    cursor.execute("INSERT INTO prox_bank VALUES (?, ?)", 
                   (row['NPA'], row['2023']))

for _, row in employment.iterrows():
    cursor.execute("INSERT INTO employment VALUES (?, ?)", 
                   (row['NPA'], row['2023']))

for _, row in education.iterrows():
    cursor.execute("INSERT INTO education VALUES (?, ?)", 
                   (row['NPA'], row['2023']))
for _, row in age.iterrows():
    cursor.execute("INSERT INTO age VALUES (?, ?)", 
                   (row['NPA'], row['2023']))
for _, row in home_owner.iterrows():
    cursor.execute("INSERT INTO home_owner VALUES (?, ?)", 
                   (row['NPA'], row['2023']))

In [565]:
def clean_npa(df):
    df['NPA'] = (
        df['NPA']
        .astype(str)
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
    )
    return df

income = clean_npa(income)
prox_bank = clean_npa(prox_bank)
employment = clean_npa(employment)
age = clean_npa(age)
home_owner = clean_npa(home_owner)
education = clean_npa(education)

In [566]:
query = """
SELECT 
    i.NPA,
    i.median_income AS income,
    pb."2023" AS prox_bank,
    e.employment_rate AS employment,
    ed.bachelors_percent,
    ag.median_age,
    ho.home_owner_percent
FROM income i
LEFT JOIN prox_bank pb 
    ON TRIM(CAST(i.NPA AS TEXT)) = TRIM(CAST(pb.NPA AS TEXT))
LEFT JOIN employment e 
    ON TRIM(CAST(i.NPA AS TEXT)) = TRIM(CAST(e.NPA AS TEXT))
LEFT JOIN education ed 
    ON TRIM(CAST(i.NPA AS TEXT)) = TRIM(CAST(ed.NPA AS TEXT))
LEFT JOIN age ag 
    ON TRIM(CAST(i.NPA AS TEXT)) = TRIM(CAST(ag.NPA AS TEXT))
LEFT JOIN home_owner ho 
    ON TRIM(CAST(i.NPA AS TEXT)) = TRIM(CAST(ho.NPA AS TEXT))
"""

result = pd.read_sql_query(query, conn)
print(result.head())
print(result.info())

   NPA    income prox_bank employment bachelors_percent median_age  \
0    2   75084.0     0.245      0.956             0.352       33.0   
1    3  117630.0       1.0      0.974             0.854       30.0   
2    4  250001.0     0.152      0.943             0.894       43.0   
3    5   49539.0      0.19      0.828             0.025       30.0   
4    6   37907.0     0.699        1.0              0.21       42.0   

  home_owner_percent  
0              0.403  
1              0.392  
2                1.0  
3              0.149  
4              0.373  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 459 entries, 0 to 458
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   NPA                 459 non-null    int64 
 1   income              459 non-null    object
 2   prox_bank           459 non-null    object
 3   employment          459 non-null    object
 4   bachelors_percent   459 non-null    object
 

In [567]:
#Classification model starts here

In [568]:
df = result.copy()

# target
df['income'] = pd.to_numeric(df['income'], errors='coerce')
df['high_income'] = (df['income'] > df['income'].median()).astype(int)

In [569]:
X = df[['prox_bank', 'employment', 'bachelors_percent', 'median_age', 'home_owner_percent']]

# Target
y = df['high_income']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=60
)

In [570]:
X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]

In [572]:
X_train.iloc[:, 0].apply(type).value_counts()

prox_bank
<class 'float'>    365
<class 'str'>        2
Name: count, dtype: int64

In [573]:
X_train = X_train.apply(pd.to_numeric, errors='coerce')
X_test = X_test.apply(pd.to_numeric, errors='coerce')

In [574]:
print(X_train.dtypes)

prox_bank             float64
employment            float64
bachelors_percent     float64
median_age            float64
home_owner_percent    float64
dtype: object


In [577]:
X_train = X_train.dropna()
y_train = y_train.loc[X_train.index]

In [578]:
tree = DecisionTreeClassifier(max_depth=5)
tree.fit(X_train, y_train)

y_pred_tree = tree.predict(X_test)

print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_tree))

Decision Tree Accuracy: 0.8043478260869565


In [580]:
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

X_test = X_test.fillna(X_train.median())
y_pred_log = log_model.predict(X_test)


print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log))

Logistic Regression Accuracy: 0.8369565217391305


In [ ]:
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)

y_pred_lda = lda.predict(X_test)

print("LDA Accuracy:", accuracy_score(y_test, y_pred_lda))

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)

print("kNN Accuracy:", accuracy_score(y_test, y_pred_knn))

In [ ]:
from pygam import LogisticGAM

gam = LogisticGAM()
gam.fit(X_train, y_train)

y_pred_gam = gam.predict(X_test)

print("GAM Accuracy:", accuracy_score(y_test, y_pred_gam))

In [ ]:
print("""
Model Comparison:
-----------------
Decision Tree:      {}
Logistic Regression: {}
LDA:                {}
kNN:                {}
GAM:                {}
""".format(
    accuracy_score(y_test, y_pred_tree),
    accuracy_score(y_test, y_pred_log),
    accuracy_score(y_test, y_pred_lda),
    accuracy_score(y_test, y_pred_knn),
    accuracy_score(y_test, y_pred_gam)
))